In [1]:
import ollama

response = ollama.chat(
    model="mistral",
    messages=[{"role": "user", "content": "Responde solo con este JSON sin texto adicional: {\"score\": 75, \"justificacion\": \"prueba exitosa\"}"}]
)
print(response["message"]["content"])

 {"score": 75, "justificacion": "prueba exitosa"}


In [3]:
from pathlib import Path
# 03_llm_scoring.ipynb

import os
import glob
import json
import time
from datetime import datetime

import pandas as pd
import ollama


# ── CONFIGURACIÓN ────────────────────────────────────

PROJECT_ROOT = str(Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent)
os.chdir(PROJECT_ROOT)

MODEL_NAME = "mistral"
BUY_BOX_PATH = "config/buy_box_malaga_2026.md"

INPUT_PATTERN = "data/processed/activos_*.csv"
OUTPUT_DIR = "data/output"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ── CARGAR DATASET DE ACTIVOS ─────────────────────────

archivos = sorted(glob.glob(INPUT_PATTERN))

if not archivos:
    raise FileNotFoundError(f"No se encontraron archivos con patrón: {INPUT_PATTERN}")

input_file = archivos[-1]
df = pd.read_csv(input_file)

print(f"Archivo cargado: {input_file}")
print(f"Pisos a evaluar: {len(df)}")


# ── VALIDAR COLUMNAS NECESARIAS ───────────────────────

required_columns = [
    "url",
    "titulo",
    "ubicacion",
    "precio",
    "m2",
    "habitaciones",
    "baños",
    "planta",
    "ascensor",
    "tipo",
    "estado",
    "año",
    "comentario",
    "plataforma"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Faltan columnas necesarias: {missing_columns}")


# ── CARGAR BUY BOX ────────────────────────────────────

with open(BUY_BOX_PATH, "r", encoding="utf-8") as f:
    BUY_BOX = f.read()

print(f"Buy Box cargado desde: {BUY_BOX_PATH}")


# ── HELPERS ───────────────────────────────────────────

def limpiar_json(texto):
    """
    Extrae el primer objeto JSON válido aproximado desde la respuesta del modelo.
    """
    inicio = texto.find("{")
    fin = texto.rfind("}") + 1

    if inicio == -1 or fin == 0:
        raise ValueError("No JSON object found in model response")

    json_limpio = texto[inicio:fin]
    json_limpio = json_limpio.replace("True", "true")
    json_limpio = json_limpio.replace("False", "false")
    json_limpio = json_limpio.replace("None", "null")

    return json_limpio


def normalizar_recomendacion(valor):
    """
    Estandariza las recomendaciones permitidas.
    """
    if pd.isna(valor):
        return "error"

    valor = str(valor).lower().strip()

    mapping = {
        "descartado": "descartar",
        "descartada": "descartar",
        "descartados": "descartar",
        "descartar": "descartar",
        "vale visita seria": "vale visita",
        "visitar": "vale visita",
        "oportunidad": "oportunidad fuerte",
        "oportunidad fuerte": "oportunidad fuerte",
        "solo si precio excelente": "solo si precio excelente"
    }

    return mapping.get(valor, valor)


def limitar_score(score):
    """
    Asegura que el score esté entre 0 y 100.
    """
    try:
        score = float(score)
        return max(0, min(100, score))
    except:
        return None


def score_piso(piso):
    """
    Envía una vivienda al modelo local vía Ollama y devuelve un JSON parseado.
    """
    prompt = f"""
Eres un analista inmobiliario especializado en Málaga.

Evalúa la siguiente vivienda usando estrictamente el BUY BOX proporcionado.

Reglas:
- Responde SOLO con JSON válido.
- No añadas texto antes ni después.
- No inventes datos.
- Si falta información crítica, indícalo en missing_critical_info.
- Si hay kill criteria, recomendación debe ser "descartar".
- El score_total debe estar entre 0 y 100.
- La recomendación debe ser una de estas:
  - oportunidad fuerte
  - vale visita
  - solo si precio excelente
  - descartar

BUY BOX:
{BUY_BOX}

VIVIENDA:
- Título: {piso.get('titulo', '')}
- Ubicación: {piso.get('ubicacion', '')}
- Precio: {piso.get('precio', '')}€
- m²: {piso.get('m2', '')}
- Habitaciones: {piso.get('habitaciones', '')}
- Baños: {piso.get('baños', '')}
- Planta: {piso.get('planta', '')}
- Ascensor: {piso.get('ascensor', '')}
- Tipo: {piso.get('tipo', '')}
- Estado: {piso.get('estado', '')}
- Año construcción: {piso.get('año', '')}
- Comentario: {str(piso.get('comentario', ''))[:800]}

JSON esperado:
{{
  "kill_criteria": false,
  "kill_razon": null,
  "score_ubicacion": 0,
  "score_patrimonio": 0,
  "score_distribucion": 0,
  "score_estado": 0,
  "score_luz": 0,
  "bonus": 0,
  "penalties": 0,
  "score_total": 0,
  "recomendacion": "",
  "justificacion": "",
  "data_quality_notes": "",
  "missing_critical_info": []
}}
"""

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    raw_response = response["message"]["content"]
    json_limpio = limpiar_json(raw_response)
    resultado = json.loads(json_limpio)

    resultado["_raw_model_response"] = raw_response

    return resultado


# ── LOOP DE SCORING ───────────────────────────────────

resultados_scoring = []
errores = []

for i, row in df.iterrows():
    piso = row.to_dict()

    print(f"\nEvaluando {i + 1}/{len(df)}: {str(piso.get('titulo', ''))[:60]}")

    try:
        resultado = score_piso(piso)

        resultado["url"] = piso.get("url")
        resultado["titulo"] = piso.get("titulo")
        resultado["ubicacion"] = piso.get("ubicacion")
        resultado["precio"] = piso.get("precio")
        resultado["m2"] = piso.get("m2")
        resultado["habitaciones"] = piso.get("habitaciones")
        resultado["baños"] = piso.get("baños")
        resultado["planta"] = piso.get("planta")
        resultado["ascensor"] = piso.get("ascensor")
        resultado["plataforma"] = piso.get("plataforma")

        resultado["score_total"] = limitar_score(resultado.get("score_total"))
        resultado["recomendacion"] = normalizar_recomendacion(resultado.get("recomendacion"))

        resultados_scoring.append(resultado)

        print(f"✓ Score: {resultado['score_total']} — {resultado['recomendacion']}")

    except Exception as e:
        error_row = {
            "url": piso.get("url"),
            "titulo": piso.get("titulo"),
            "ubicacion": piso.get("ubicacion"),
            "precio": piso.get("precio"),
            "m2": piso.get("m2"),
            "habitaciones": piso.get("habitaciones"),
            "baños": piso.get("baños"),
            "planta": piso.get("planta"),
            "ascensor": piso.get("ascensor"),
            "plataforma": piso.get("plataforma"),
            "score_total": None,
            "recomendacion": "error",
            "justificacion": str(e)
        }

        resultados_scoring.append(error_row)
        errores.append(error_row)

        print(f"✗ Error: {e}")

    time.sleep(1)


# ── CREAR DATAFRAMES ──────────────────────────────────

df_scoring = pd.DataFrame(resultados_scoring)
df_errores = pd.DataFrame(errores)

df_scoring["recomendacion"] = df_scoring["recomendacion"].apply(normalizar_recomendacion)
df_scoring["score_total"] = df_scoring["score_total"].apply(limitar_score)

df_scoring = df_scoring.sort_values("score_total", ascending=False, na_position="last")


# ── GUARDAR OUTPUTS ───────────────────────────────────

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

ranking_path = f"{OUTPUT_DIR}/ranking_final_{timestamp}.csv"
errors_path = f"{OUTPUT_DIR}/scoring_errors_{timestamp}.csv"

df_scoring.to_csv(ranking_path, index=False)

if not df_errores.empty:
    df_errores.to_csv(errors_path, index=False)


# ── REPORTE FINAL ─────────────────────────────────────

print("\nScoring completo")
print(f"Pisos procesados: {len(df_scoring)}")
print(f"Errores: {len(df_errores)}")

print("\nRecomendaciones:")
print(df_scoring["recomendacion"].value_counts(dropna=False))

print("\nTop 10:")
display_cols = [
    "titulo",
    "precio",
    "m2",
    "habitaciones",
    "score_total",
    "recomendacion",
    "plataforma"
]

print(df_scoring[display_cols].head(10).to_string(index=False))

print(f"\nRanking guardado en: {ranking_path}")

if not df_errores.empty:
    print(f"Errores guardados en: {errors_path}")

Archivo cargado: data/processed/activos_20260524_1341.csv
Pisos a evaluar: 136


FileNotFoundError: [Errno 2] No such file or directory: 'config/buy_box_malaga_2026.md'